# UniLumos BDS Official All-Demos Certificate

Colab-first notebook for the calibration-to-holdout BSS Deployment Score analysis on existing UniLumos official-demo results.

This notebook is analysis-only:
- does not rerun the full UniLumos benchmark
- does not train anything
- does not change weights
- does not overwrite the BSS run outputs

Large benchmark artifacts stay on Google Drive. Code is cloned or pulled from GitHub into the Colab runtime, then this notebook reads the Drive result folders and writes the BDS certificate back to Drive.

## 1. Configure GitHub, Drive Paths, And Run Flags

Set `BRANCH` to the branch that contains the BDS scripts before running the clone/pull cell. Keep `RUN_COMPUTE_BDS = True`; the notebook will automatically skip BDS scoring if the gain table cannot be built from existing metrics.

In [ ]:
from pathlib import Path
import os
import sys
import subprocess
import shutil
import json
import csv
from textwrap import indent

IN_COLAB = Path("/content").exists()

GITHUB_REPO = "https://github.com/WANG-Ruipeng/Lumos-Custom.git"
BRANCH = "bss-unilumos-official-smoke"
LOCAL_REPO_ROOT = Path("/content/Lumos-Custom") if IN_COLAB else Path.cwd()

DRIVE_ROOT = Path("/content/drive/MyDrive") if IN_COLAB else Path.cwd()
BSS_RUN_ROOTS = [
    DRIVE_ROOT / "Colab_Projects" / "UniLumos-BSS-Runs" / "model_b_unilumos_official_all_demos_bss_v1",
    Path("/content/UniLumos-BSS-Runs/model_b_unilumos_official_all_demos_bss_v1"),
    DRIVE_ROOT / "Colab_Projects" / "UniLumos-BSS-Runs" / "model_b_unilumos_official_bss_smoke_v1",
]

BDS_OUTPUT_ROOT = DRIVE_ROOT / "Colab_Projects" / "UniLumos-BDS" / "unilumos_bds_certificate_v1"
BOOTSTRAP_RESAMPLES = 10000
RUN_GIT_PULL = True
RUN_BUILD_GAIN_TABLE = True
RUN_COMPUTE_BDS = True

print("IN_COLAB =", IN_COLAB)
print("GITHUB_REPO =", GITHUB_REPO)
print("BRANCH =", BRANCH)
print("LOCAL_REPO_ROOT =", LOCAL_REPO_ROOT)
print("DRIVE_ROOT =", DRIVE_ROOT)
print("BDS_OUTPUT_ROOT =", BDS_OUTPUT_ROOT)
print("BOOTSTRAP_RESAMPLES =", BOOTSTRAP_RESAMPLES)
for root in BSS_RUN_ROOTS:
    print("BSS_RUN_ROOT =", root)

## 2. Mount Google Drive

Drive contains the official-demo BSS result folders and receives the BDS certificate outputs.

In [ ]:
if IN_COLAB:
    from google.colab import drive
    drive.mount("/content/drive")
    assert DRIVE_ROOT.exists(), DRIVE_ROOT
else:
    print("Not running in Colab; using local filesystem paths from the configuration cell.")

## 3. Clone Or Pull The Experiment Branch

Rerun this cell after pushing updated code to GitHub. The notebook expects the UniLumos code tree at `LOCAL_REPO_ROOT / "UniLumos"`.

In [ ]:
def run(cmd, cwd=None, check=True):
    print("$", " ".join(str(x) for x in cmd))
    return subprocess.run(cmd, cwd=cwd, check=check, text=True)

if IN_COLAB:
    if LOCAL_REPO_ROOT.exists() and (LOCAL_REPO_ROOT / ".git").exists():
        if RUN_GIT_PULL:
            run(["git", "fetch", "origin", BRANCH], cwd=LOCAL_REPO_ROOT)
            run(["git", "checkout", BRANCH], cwd=LOCAL_REPO_ROOT)
            run(["git", "pull", "--ff-only", "origin", BRANCH], cwd=LOCAL_REPO_ROOT)
        else:
            print("RUN_GIT_PULL is False; using existing local checkout.")
    else:
        LOCAL_REPO_ROOT.parent.mkdir(parents=True, exist_ok=True)
        run(["git", "clone", "-b", BRANCH, GITHUB_REPO, str(LOCAL_REPO_ROOT)])
else:
    current = Path.cwd().resolve()
    for candidate in [current, *current.parents]:
        if (candidate / ".git").exists():
            LOCAL_REPO_ROOT = candidate
            break

TASK_ROOT = LOCAL_REPO_ROOT / "UniLumos"
CODE_ROOT = TASK_ROOT / "UniLumos"
assert TASK_ROOT.exists(), TASK_ROOT
assert (CODE_ROOT / "unilumos_infer_abc.py").exists(), CODE_ROOT

os.environ["PYTHONPATH"] = f"{TASK_ROOT}:{CODE_ROOT}:" + os.environ.get("PYTHONPATH", "")
for path in [TASK_ROOT, CODE_ROOT]:
    if str(path) not in sys.path:
        sys.path.insert(0, str(path))

commit = subprocess.check_output(["git", "rev-parse", "--short", "HEAD"], cwd=LOCAL_REPO_ROOT, text=True).strip()
print("LOCAL_REPO_ROOT =", LOCAL_REPO_ROOT)
print("TASK_ROOT =", TASK_ROOT)
print("CODE_ROOT =", CODE_ROOT)
print("Checked out commit =", commit)

## 4. Verify BDS Scripts And Archive Them Into The Certificate Folder

The scripts live in GitHub under `UniLumos/bss_experiments/unilumos_bds_certificate_v1/scripts/`. This cell copies those scripts into the Drive certificate folder so the certificate is self-contained.

In [ ]:
BDS_SCRIPT_ROOT = TASK_ROOT / "bss_experiments" / "unilumos_bds_certificate_v1" / "scripts"
BUILD_GAIN_SCRIPT = BDS_SCRIPT_ROOT / "build_same_compute_gain_table.py"
COMPUTE_BDS_SCRIPT = BDS_SCRIPT_ROOT / "compute_bds.py"

required_scripts = [BUILD_GAIN_SCRIPT, COMPUTE_BDS_SCRIPT]
for script in required_scripts:
    print(script, script.exists())
    assert script.exists(), script

BDS_OUTPUT_ROOT.mkdir(parents=True, exist_ok=True)
(BDS_OUTPUT_ROOT / "scripts").mkdir(parents=True, exist_ok=True)
for script in required_scripts:
    shutil.copy2(script, BDS_OUTPUT_ROOT / "scripts" / script.name)

print("Archived scripts to", BDS_OUTPUT_ROOT / "scripts")

## 5. Audit Existing BSS Result Folders

This is a lightweight preflight. The formal audit report is written by the next cell to `reports/00_artifact_audit.md`.

In [ ]:
EXPECTED_RELATIVE_FILES = [
    "metrics/master_long_metrics.csv",
    "metrics/metrics_against_ref.csv",
    "metrics/per_case_metrics.csv",
    "tables/table1a_same_compute_rgb_closure_main.csv",
    "tables/table1b_same_compute_rgb_closure_support_aware.csv",
    "tables/table2_matched_quality_compute_saving.csv",
    "tables/table3_same_nfe_main.csv",
    "tables/table4_failure_or_smallest_gain_cases.csv",
    "tables/cross_model_summary_row.csv",
]

for root in BSS_RUN_ROOTS:
    print("\n", root)
    print("exists =", root.exists())
    if root.exists():
        for rel in EXPECTED_RELATIVE_FILES:
            print(" ", rel, (root / rel).exists())
        manifests = sorted((root / "manifests").glob("*.csv")) if (root / "manifests").exists() else []
        print(" manifests/*.csv =", len(manifests))

## 6. Phase 0-1: Artifact Audit And Same-Compute Gain Table

This runs only analysis over existing result CSVs. It uses exact same-compute `uniformT`/`bssT` pairs and writes `metrics/unilumos_same_compute_gain_long.csv` only if RGB-L1 closure can be recovered per case.

In [ ]:
gain_table_path = BDS_OUTPUT_ROOT / "metrics" / "unilumos_same_compute_gain_long.csv"

if RUN_BUILD_GAIN_TABLE:
    build_cmd = [
        sys.executable,
        str(BUILD_GAIN_SCRIPT),
        "--repo-root", str(TASK_ROOT),
        "--output-root", str(BDS_OUTPUT_ROOT),
    ]
    for root in BSS_RUN_ROOTS:
        build_cmd.extend(["--input-root", str(root)])
    print("$", " ".join(build_cmd))
    build_result = subprocess.run(build_cmd, text=True, check=False)
    print("build_same_compute_gain_table returncode =", build_result.returncode)
else:
    print("RUN_BUILD_GAIN_TABLE is False; using existing gain table if present.")

print("gain_table_path =", gain_table_path, gain_table_path.exists())
if not gain_table_path.exists():
    audit_path = BDS_OUTPUT_ROOT / "reports" / "00_artifact_audit.md"
    print("BDS computation is blocked because the gain table does not exist.")
    print("artifact_audit =", audit_path, audit_path.exists())
    if audit_path.exists():
        print("\n" + "=" * 80)
        print(audit_path.read_text(encoding="utf-8")[:5000])

## 7. Phase 2-6: Splits, BDS, Holdout Verification, And Final Report

This cell runs only if the gain table exists. Bootstrap resampling is case-level and does not interpolate NFE points.

In [ ]:
if RUN_COMPUTE_BDS and gain_table_path.exists():
    bds_cmd = [
        sys.executable,
        str(COMPUTE_BDS_SCRIPT),
        "--repo-root", str(TASK_ROOT),
        "--output-root", str(BDS_OUTPUT_ROOT),
        "--bootstrap-resamples", str(BOOTSTRAP_RESAMPLES),
    ]
    print("$", " ".join(bds_cmd))
    run(bds_cmd, check=True)
else:
    print("Skipping compute_bds.py")
    print("RUN_COMPUTE_BDS =", RUN_COMPUTE_BDS)
    print("gain table exists =", gain_table_path.exists())

## 8. Print Certificate Paths And Previews

Use this after the analysis cells finish. If artifacts are missing, this cell still prints the blocked audit and final report paths.

In [ ]:
paths = {
    "artifact_audit": BDS_OUTPUT_ROOT / "reports" / "00_artifact_audit.md",
    "same_compute_gain_table": BDS_OUTPUT_ROOT / "metrics" / "unilumos_same_compute_gain_long.csv",
    "bds_split_table": BDS_OUTPUT_ROOT / "tables" / "tableA_unilumos_bds_by_split.csv",
    "cross_model_bds_row": BDS_OUTPUT_ROOT / "tables" / "cross_model_bds_row.csv",
    "bds_vs_full_summary": BDS_OUTPUT_ROOT / "tables" / "tableB_bds_vs_full_summary.csv",
    "final_report": BDS_OUTPUT_ROOT / "reports" / "FINAL_UNILUMOS_BDS_CERTIFICATE_REPORT.md",
}

for name, path in paths.items():
    print(f"{name}: {path} exists={path.exists()}")

final_report = paths["final_report"]
if final_report.exists():
    print("\n" + "=" * 80)
    print(final_report.read_text(encoding="utf-8")[:8000])

split_table = paths["bds_split_table"]
if split_table.exists():
    print("\n" + "=" * 80)
    print("BDS split table preview")
    with split_table.open("r", encoding="utf-8", newline="") as handle:
        reader = csv.DictReader(handle)
        for index, row in enumerate(reader):
            print(row)
            if index >= 5:
                break